# TUGAS PRAKTIKUM #1

**Nama : Diva Hersa**  
**NIM : 2411513034**  

**Mata Kuliah:** Pengantar Pemrosesan Paralel  
**Latihan Mandiri**

---


## 1. Klasifikasi Flynn

### a. Operasi `arr + arr` pada NumPy array

**Kategori: SIMD (Single Instruction, Multiple Data)**

Alasan: operasi yang sama diterapkan pada banyak elemen array secara bersamaan. NumPy dapat memanfaatkan operasi vektor/SIMD pada level perangkat keras.

### b. Program Python dengan threading/multiprocessing membaca file dan memproses baris demi baris

**Kategori: MIMD (Multiple Instruction, Multiple Data)**

Alasan: beberapa proses/thread dapat bekerja secara independen pada bagian data masing-masing. Pada multiprocessing, setiap proses memiliki alur instruksi dan data sendiri.

### c. Cluster komputer dengan setiap node menjalankan simulasi berbeda

**Kategori: MIMD (Multiple Instruction, Multiple Data)**

Alasan: setiap node dapat menjalankan instruksi atau simulasi yang berbeda dengan data yang berbeda secara independen.


### Ringkasan Klasifikasi

| Skenario | Kategori Flynn | Alasan |
|---|---|---|
| `arr + arr` NumPy | **SIMD** | Satu operasi diterapkan pada banyak data |
| Threading/multiprocessing | **MIMD** | Beberapa proses/thread dapat bekerja secara independen |
| Cluster simulasi berbeda | **MIMD** | Setiap node dapat menjalankan instruksi dan data berbeda |


## 2. Analisis Hukum Amdahl

Diketahui bagian program yang dapat diparalelkan adalah **85%**.

$$P = 0.85$$

Bagian serial:

$$1-P = 0.15$$

Rumus Hukum Amdahl:

$$S(N)=\frac{1}{(1-P)+\frac{P}{N}}$$


In [ ]:
import matplotlib.pyplot as plt

P = 0.85
N = [2, 4, 8, 16, 32, 64]

speedup = []

for n in N:
    S = 1 / ((1 - P) + (P / n))
    speedup.append(S)
    print(f"N = {n:2d} -> Speedup = {S:.4f}")


### Hasil Perhitungan

Hasil perhitungan berdasarkan rumus Hukum Amdahl adalah sebagai berikut:

| Jumlah Prosesor (N) | Speedup |
|---:|---:|
| 2 | 1.7391 |
| 4 | 2.7586 |
| 8 | 3.9024 |
| 16 | 4.9231 |
| 32 | 5.5103 |
| 64 | 5.7574 |

Nilai tersebut akan dihitung kembali secara otomatis oleh program saat kode dijalankan.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(N, speedup, marker='o')
plt.xlabel("Jumlah Prosesor (N)")
plt.ylabel("Speedup")
plt.title("Speedup berdasarkan Hukum Amdahl")
plt.grid(True)
plt.show()


### Analisis

Dari hasil perhitungan dapat dilihat bahwa semakin banyak prosesor yang digunakan, nilai *speedup* juga semakin besar. Namun, peningkatannya tidak terus bertambah secara besar. Hal ini karena masih ada 15% bagian program yang tidak dapat diparalelkan. Jadi, penambahan jumlah prosesor memiliki batas dalam meningkatkan kecepatan program.

Batas maksimum *speedup* adalah:

$$S_{max}=\frac{1}{1-P}=\frac{1}{0.15}=6.67$$

Artinya, secara teori program tersebut tidak dapat menjadi lebih cepat dari sekitar 6,67 kali walaupun jumlah prosesor terus ditambah.


## 3. Benchmark CPU-Bound

Benchmark menggunakan fungsi CPU-bound untuk menghitung jumlah bilangan prima sampai batas tertentu. Waktu pengujian diukur dengan `time.perf_counter()` dan `timeit`.

In [ ]:
def hitung_prima(N):
    jumlah = 0

    for n in range(2, N + 1):
        prima = True

        for i in range(2, int(n ** 0.5) + 1):
            if n % i == 0:
                prima = False
                break

        if prima:
            jumlah += 1

    return jumlah


In [ ]:
import time

ukuran_input = [1000, 2000, 4000, 8000]
hasil_perf = []

for N in ukuran_input:
    start = time.perf_counter()
    hasil = hitung_prima(N)
    end = time.perf_counter()

    waktu = end - start
    hasil_perf.append(waktu)

    print(f"N = {N:5d} | Prima = {hasil:5d} | Waktu = {waktu:.6f} detik")


In [ ]:
import timeit

hasil_timeit = []

for N in ukuran_input:
    waktu = timeit.timeit(
        lambda: hitung_prima(N),
        number=3
    ) / 3

    hasil_timeit.append(waktu)
    print(f"N = {N:5d} | Waktu rata-rata = {waktu:.6f} detik")


In [ ]:
print("Perbandingan hasil benchmark")
print("-" * 60)

for i in range(len(ukuran_input)):
    print(
        f"N = {ukuran_input[i]:5d} | "
        f"perf_counter = {hasil_perf[i]:.6f} s | "
        f"timeit = {hasil_timeit[i]:.6f} s"
    )


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(ukuran_input, hasil_perf, marker='o', label='time.perf_counter()')
plt.plot(ukuran_input, hasil_timeit, marker='s', label='timeit')
plt.xlabel("Ukuran Input (N)")
plt.ylabel("Waktu Eksekusi (detik)")
plt.title("Perbandingan Benchmark CPU-Bound")
plt.legend()
plt.grid(True)
plt.show()


### Analisis Benchmark

Berdasarkan hasil percobaan, waktu eksekusi bertambah ketika nilai input semakin besar. Hal ini terjadi karena program harus memeriksa lebih banyak bilangan untuk menentukan apakah bilangan tersebut merupakan bilangan prima. Hasil pengukuran menggunakan `time.perf_counter()` dan `timeit` dapat sedikit berbeda karena kondisi komputer saat program dijalankan. Namun, secara umum keduanya menunjukkan bahwa semakin besar input, semakin lama waktu yang diperlukan.


## 4. Refleksi Singkat

Hukum Amdahl lebih relevan digunakan ketika bagian program yang dapat diparalelkan sudah diketahui dan ukuran masalah relatif tetap. Hukum ini membantu mengetahui batas maksimum peningkatan kinerja akibat penambahan prosesor. Semakin besar bagian serial dalam program, semakin terbatas *speedup* yang dapat diperoleh. Sementara itu, Hukum Gustafson lebih sesuai ketika ukuran masalah dapat diperbesar seiring bertambahnya jumlah prosesor. Dalam kondisi tersebut, prosesor tambahan dapat digunakan untuk mengerjakan masalah yang lebih besar sehingga pemanfaatan sistem paralel dapat meningkat. Pemilihan Hukum Amdahl atau Gustafson bergantung pada kondisi masalah, terutama apakah ukuran masalah tetap atau ikut berkembang ketika jumlah prosesor ditambah.